# Task 8: Intelligent Retraining Strategy
**Assignment 4 — IEEE CIS Fraud Detection**

Compare three retraining policies:
1. **Threshold-based** — retrain when rolling recall < 0.70
2. **Periodic** — retrain every N batches
3. **Hybrid** — periodic + threshold override

Evaluation criteria: mean recall, stability (std), retrain count (compute cost proxy), final performance.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import roc_auc_score, recall_score
import warnings
warnings.filterwarnings('ignore')

from src.data.ingest import load_sample
from src.data.preprocess import preprocess
from src.drift.time_split import quartile_split, inject_new_fraud_pattern

os.makedirs('../outputs/retraining', exist_ok=True)
print('Libraries loaded')

In [ ]:
# Load data and simulate batched streaming
df = load_sample()
q12, q3, q4 = quartile_split(df)
q4_drift = inject_new_fraud_pattern(q4, fraud_injection_rate=0.25)

# Initial train
X_train, y_train, enc, med = preprocess(q12, fit=True)

# Q4 split into 10 batches to simulate streaming
n_batches = 10
batch_size = len(q4_drift) // n_batches
batches_raw = [q4_drift.iloc[i*batch_size:(i+1)*batch_size] for i in range(n_batches)]
batches = [(preprocess(b, encoders=enc, medians=med, fit=False)[:2]) for b in batches_raw]

print(f'Batches: {n_batches} x {batch_size} rows')

In [ ]:
def train_xgb(X_tr, y_tr):
    neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
    spw = neg / max(pos, 1)
    m = xgb.XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        scale_pos_weight=spw, tree_method='hist',
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
    m.fit(X_tr.values, y_tr.values, verbose=False)
    return m

def eval_batch(model, X_b, y_b, thresh=0.5):
    proba = model.predict_proba(X_b.values)[:, 1]
    y_pred = (proba >= thresh).astype(int)
    return recall_score(y_b.values, y_pred, zero_division=0), roc_auc_score(y_b.values, proba)

print('Helper functions defined')

In [ ]:
RECALL_THRESHOLD = 0.70
PERIODIC_INTERVAL = 3   # retrain every 3 batches

results = {}

for strategy in ['threshold', 'periodic', 'hybrid']:
    model = train_xgb(X_train, y_train)
    recalls, aucs, retrain_batches = [], [], []
    cumX = X_train.copy()
    cumY = y_train.copy()

    for i, (X_b, y_b) in enumerate(batches):
        recall, auc = eval_batch(model, X_b, y_b)
        recalls.append(recall)
        aucs.append(auc)

        # Accumulate new data
        cumX = pd.concat([cumX, X_b], ignore_index=True)
        cumY = pd.concat([cumY, y_b], ignore_index=True)

        do_retrain = False
        if strategy == 'threshold':
            do_retrain = recall < RECALL_THRESHOLD
        elif strategy == 'periodic':
            do_retrain = (i + 1) % PERIODIC_INTERVAL == 0
        elif strategy == 'hybrid':
            do_retrain = (recall < RECALL_THRESHOLD) or ((i + 1) % PERIODIC_INTERVAL == 0)

        if do_retrain:
            model = train_xgb(cumX, cumY)
            retrain_batches.append(i)
            print(f'  [{strategy}] Retrain at batch {i+1} (recall={recall:.3f})')

    results[strategy] = {
        'recalls': recalls, 'aucs': aucs,
        'retrain_count': len(retrain_batches),
        'retrain_batches': retrain_batches,
        'mean_recall': np.mean(recalls),
        'std_recall': np.std(recalls),
        'final_recall': recalls[-1],
        'final_auc': aucs[-1],
    }
    print(f'[{strategy}] mean_recall={np.mean(recalls):.3f}  retrains={len(retrain_batches)}')

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Strategy':      list(results.keys()),
    'Mean Recall':   [round(v['mean_recall'], 3)  for v in results.values()],
    'Std Recall':    [round(v['std_recall'], 3)   for v in results.values()],
    'Final Recall':  [round(v['final_recall'], 3) for v in results.values()],
    'Final AUC':     [round(v['final_auc'], 3)    for v in results.values()],
    'Retrain Count': [v['retrain_count']           for v in results.values()],
})
print('\n=== Retraining Strategy Comparison ===')
print(summary.to_string(index=False))
summary.to_csv('../outputs/retraining/strategy_comparison.csv', index=False)

In [ ]:
# Recall over batches plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'threshold': 'steelblue', 'periodic': 'darkorange', 'hybrid': 'green'}

for name, res in results.items():
    axes[0].plot(res['recalls'], label=name, color=colors[name], marker='o')
    axes[1].plot(res['aucs'],    label=name, color=colors[name], marker='s')
    for rb in res['retrain_batches']:
        axes[0].axvline(rb, color=colors[name], ls='--', alpha=0.4)

axes[0].axhline(RECALL_THRESHOLD, color='red', ls=':', label='Recall threshold')
axes[0].set_title('Fraud Recall per Batch')
axes[0].set_xlabel('Batch'); axes[0].set_ylabel('Recall')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('AUC-ROC per Batch')
axes[1].set_xlabel('Batch'); axes[1].set_ylabel('AUC-ROC')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Retraining Strategy Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/retraining/strategy_comparison.png', dpi=150)
plt.show()
print('Saved → ../outputs/retraining/strategy_comparison.png')